# TD et TME 9: CREATION DE SCHEMAS-MODIFICATION DES DONNEES


# Installation H2 

Le système utilisé pendant les TME est H2.

In [ ]:
pip install psycopg2-binary

Télécharger le pilote de H2

**Relancez le kernel**: Kernel -> Restart kernel ...

https://www.psycopg.org/docs/
http://localhost:8082

In [2]:
import psycopg2
import os
local_dir = "./data"
os.makedirs(local_dir, exist_ok=True)
os.listdir(local_dir)

[]

**Attention**: vous pouvez cliquer sur les 3 lignes dans la fenêtre de gauche pour d'afficher les différentes sections du notebook   

# Démarrage du serveur H2
Démarrer un serveur de base de données H2 en arrière-plan, accessible uniquement localement(localhost, 127.0.0.1) sur le port 8082, avec les fichiers des bases de données stockés dans ./data (un fichier *.mv.db par base de données)

In [3]:
port = 8082
print(f'Le numero du port utilisé pour la connexion à la BD est: {port}')

Le numero du port utilisé pour la connexion à la BD est: 8082


In [4]:
%%bash --bg --out output -s "$port"
java -Dh2.bindAddress=127.0.0.1 -cp h2-2.1.214.jar:postgresql-42.6.0.jar org.h2.tools.Server -pg -pgPort $1 -baseDir ./data -ifNotExists

Couldn't find program: 'bash'


Tester si le serveur a été démarré

In [5]:
%%bash
nc -zv 127.0.0.1 8082

Couldn't find program: 'bash'


In [6]:
import psycopg2
import os
import socket
import subprocess
import time

In [7]:
local_dir = "./data"
os.makedirs(local_dir, exist_ok=True)
os.listdir(local_dir)

[]

In [8]:
port = 8082
print(f"Le numero du port utilisé pour la connexion à la BD est: {port}")

Le numero du port utilisé pour la connexion à la BD est: 8082


In [9]:
cmd = [
    "java",
    "-Dh2.bindAddress=127.0.0.1",
    "-cp",
    "h2-2.1.214.jar;postgresql-42.6.0.jar",
    "org.h2.tools.Server",
    "-pg",
    "-pgPort",
    str(port),
    "-baseDir",
    "./data",
    "-ifNotExists"
]

h2_process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

time.sleep(2)

if h2_process.poll() is None:
    print("H2 server started successfully in background.")
else:
    out, err = h2_process.communicate()
    print("H2 server failed to start.")
    print("STDOUT:\n", out)
    print("STDERR:\n", err)

H2 server started successfully in background.


In [10]:
s = socket.socket()
result = s.connect_ex(("127.0.0.1", port))
s.close()

if result == 0:
    print(f"Server is running on port {port}")
else:
    print(f"Server is NOT running on port {port}")

Server is running on port 8082


# Connexion du client au serveur H2
Se connecter au serveur H2 précédemment lancé en utilisant un nom de base de données, un login et un mot de passe. Si une connexion existante existe, elle est fermée avant d’en créer une nouvelle. Si la base n’existe pas encore, H2 la crée automatiquement dans le répertoire de données ./data (fichier *.mv.db). La fonction retourne un objet de connexion utilisable pour exécuter des commandes SQL sur le serveur.

In [11]:
def connect_H2(db,user,password):
    global connection
    try:
        connection
    except:
        connection = None
    if connection != None:
        try:
            connection.close()
            print("Connection closed")
        except  Error as e:
            print(f"The error '{e}' occurred")
    try:
        connection = psycopg2.connect(f"dbname={db} user={user} password={password} host=localhost port={port}")
        print("Connection to H2 DB successful")
    except Exception as e:
        print(f"The error '{e}' occurred")
    return connection

In [12]:
#Connexion à la base H2 nommée "jo<port>" avec l’utilisateur et le mot de passe spécifiés. Si la base n’existe pas encore, H2 la crée automatiquement. 
#Toute connexion précédente est fermée avant d’en établir une nouvelle.
connection = connect_H2("jo"+f'{port}',"user","pass")

Connection to H2 DB successful


# Fonction execute

Exécute une commande SQL sur la connexion fournie, affiche le nombre de lignes affectées et, si demandé, présente les résultats sous forme de tableau lisible avec les noms de colonnes. Le curseur peut être fermé automatiquement après l’exécution, et toute erreur est affichée. La fonction retourne le curseur utilisé pour accéder aux résultats ou None en cas d’échec.

In [13]:
def execute(connection, query, show=True, close=True, top=10):
    try:
        cursor = connection.cursor()
        cursor.execute(query)
        if cursor.rowcount >0:
          print(cursor.rowcount,"rows,", f"display only {top} rows")
        else:
          print(cursor.rowcount,"rows")
        if show and cursor.rowcount:
            names = [desc[0] for desc in cursor.description]

            lengths={}
            for attr in names:
                lengths[attr]=len(attr)
            for ligne in cursor:
                i=0
                for attr in ligne:
                    lengths[names[i]]=max(lengths[names[i]],len(str(attr)))
                    i=i+1
            print('|',end='')
            for attr in names:
                print(str(attr).ljust(lengths[attr]),end='|')
            print()
            print('|',end='')
            for attr in names:
                print(''.ljust(lengths[attr]+1,'-'),end='')
            print()
            cursor.execute(query)
            n=0
            for ligne in cursor:
                i=0
                print('|',end='')
                for attr in ligne:
                    print(str(attr)[:lengths[names[i]]].ljust(lengths[names[i]]),end='|')
                    i+=1
                print()
                n+=1
                if n>top :
                  break
        if close:
            cursor.close()
    except Exception as e:
        print(f"The error '{e}' occurred")
        cursor=None
    return cursor
print('fonction définie')

fonction définie


# Affichage du schéma d'une table

In [14]:
def show_table(connection,table_name):

    print(f'*********** Attributs de la table : {table_name} ************')
    query=f"""
        SELECT table_name,
              column_name,
              data_type,
              column_default,
              is_nullable,
              character_maximum_length,
              numeric_precision
        FROM information_schema.columns
        where lower(table_name)  = '{table_name}'
        """
    execute(connection,query)


    print(f'\n*********** Contraintes de la table : {table_name} ************')
    query = f"""
        SELECT tc.constraint_name, 
               tc.constraint_type, 
               cc.check_clause
        FROM information_schema.table_constraints tc
        LEFT JOIN information_schema.check_constraints cc 
               ON tc.constraint_name = cc.constraint_name
        WHERE lower(tc.table_name) = '{table_name}'
        AND tc.table_schema = 'public'
        ORDER BY tc.constraint_type
        """
    execute(connection, query)
    

# Affichage du schéma global 

In [15]:
def show_schema(connection):

    print('*********** Tables ************')
    query="""
    select TABLE_NAME
    from INFORMATION_SCHEMA.TABLES
    where TABLE_SCHEMA = 'public'
    """
    execute(connection,query,show=True,top=1000)

    print('\n\n*********** Domaines ************')
    query="""
    SELECT domain_name,check_clause
    FROM information_schema.domain_constraints a left outer join
         information_schema.check_constraints b
         on a.constraint_name=b.constraint_name
    """
    execute(connection,query,top=1000)

    print('\n\n*********** Attributs ************')
    query=f"""
    SELECT c.table_name,
           c.column_name,
           c.data_type,
           c.column_default,
           c.is_nullable,
           c.character_maximum_length,
           c.numeric_precision
    FROM INFORMATION_SCHEMA.TABLES t, information_schema.columns c
    where t.table_name=c.table_name
    and t.TABLE_SCHEMA = 'public'
    """
    execute(connection,query,top=1000)

    print('\n\n*********** Contraintes d''Intégrité ************')
    query="""
    SELECT 
        tc.table_name, 
        tc.constraint_name, 
        tc.constraint_type,
        cc.check_clause
    FROM information_schema.table_constraints tc
    LEFT JOIN information_schema.check_constraints cc 
        ON tc.constraint_name = cc.constraint_name
    WHERE tc.table_schema = 'public'
    ORDER BY tc.table_name, tc.constraint_type
    """
    execute(connection,query,top=1000)
    

# Exercices TD

pour supprimer le schéma de la base courante

In [62]:
query="""
DROP ALL OBJECTS
"""
execute(connection,query,show=True)

0 rows


<cursor object at 0x000001D112AE35A0; closed: -1>

## VILLES ET PAYS
On veut créer un schéma relationnel pour stocker des informations sur des villes et des pays.



### Traduisez le schéma relationnel suivant en instructions SQL:


- Ville(<u>nom</u>, population, pays*) 
- Pays(<u>nom</u>, capitale*)

où pays est une référence vers un pays dans la table Pays et capitale est une référence vers une ville dans la table Ville.

In [64]:
query="""
create table Pays(
    nom varchar(50) primary key,
    capital varchar(50),
    constraint ft_ville foreign key(capital) references Ville
    )
"""
execute(connection,query,show=True)

0 rows


<cursor object at 0x000001D112AE2F80; closed: -1>

In [65]:
query="""
alter table Ville add constraint fk_pays  foreign key(Pays) references Pays
"""
execute(connection,query,show=True)

0 rows


<cursor object at 0x000001D112AE3CA0; closed: -1>

In [63]:
query="""
create table Ville(
    nom varchar(50) primary key ,
    population numeric(10),
    pays varchar(50)
    
    
)

"""
execute(connection,query,show=True)

0 rows


<cursor object at 0x000001D112AE3840; closed: -1>

Interroger le catalogue de données

In [66]:
show_schema(connection)

*********** Tables ************
2 rows, display only 1000 rows
|table_name|
|-----------
|ville     |
|pays      |


*********** Domaines ************
0 rows


*********** Attributs ************
5 rows, display only 1000 rows
|table_name|column_name|data_type        |column_default|is_nullable|character_maximum_length|numeric_precision|
|---------------------------------------------------------------------------------------------------------------
|ville     |nom        |character varying|None          |NO         |50                      |None             |
|ville     |population |numeric          |None          |YES        |None                    |10               |
|ville     |pays       |character varying|None          |YES        |50                      |None             |
|pays      |nom        |character varying|None          |NO         |50                      |None             |
|pays      |capital    |character varying|None          |YES        |50                      |No

### Insérez la France avec sa capitale Paris (3 millions d'habitants) dans la base de données (deux insertions).

In [68]:
query="""
insert into Ville values ('Paris',3000000,null)
"""
execute(connection,query,show=True)

1 rows, display only 10 rows
The error ''NoneType' object is not iterable' occurred


In [69]:
query="""
insert into Pays values ('France', 'Paris')
"""
execute(connection,query,show=True)

1 rows, display only 10 rows
The error ''NoneType' object is not iterable' occurred


In [72]:
query="""
update Ville set Pays = 'France' where nom  = 'Paris'
"""
execute(connection,query,show=True)

1 rows, display only 10 rows
The error ''NoneType' object is not iterable' occurred


In [82]:
query="""
select * from Ville
"""
execute(connection,query,show=True)

1 rows, display only 10 rows
|nom  |population|pays  |
|------------------------
|Paris|3000000   |France|


<cursor object at 0x000001D112AE3920; closed: -1>

In [83]:
query="""
select * from Pays
"""
execute(connection,query,show=True)

0 rows


<cursor object at 0x000001D112AE3A00; closed: -1>

### Modifiez le schéma de telle manière que la suppression d'un pays déclenche automatiquement la suppression de toutes les villes du pays.

In [85]:
query="""
alter table ville drop constraint fk_pays
alter table pays drop constraint fk_ville
"""
execute(connection,query,show=True)

The error 'Syntax error in SQL statement "\000aalter table ville drop constraint fk_pays\000a[*]alter table pays drop constraint fk_ville\000a"; SQL statement:

alter table ville drop constraint fk_pays
alter table pays drop constraint fk_ville
 [42000-214]
DETAIL:  org.h2.jdbc.JdbcSQLSyntaxErrorException: Syntax error in SQL statement "\000aalter table ville drop constraint fk_pays\000a[*]alter table pays drop constraint fk_ville\000a"; SQL statement:

alter table ville drop constraint fk_pays
alter table pays drop constraint fk_ville
 [42000-214]
' occurred


In [77]:
query="""
delete from pays where nom='France'

"""
execute(connection,query,show=True)

1 rows, display only 10 rows
The error ''NoneType' object is not iterable' occurred


In [81]:
query="""
alter table ville add constraint fk_ville foreign key(pays) references pays on delete cascade
alter table pays add constraint fk_pays  
"""
execute(connection,query,show=True)

The error 'Referential integrity constraint violation: "fk_pays: public.ville FOREIGN KEY(pays) REFERENCES public.pays(nom)"; SQL statement:

alter table ville add constraint fk_pays foreign key(pays) references pays on delete cascad [23506-214]
DETAIL:  org.h2.jdbc.JdbcSQLIntegrityConstraintViolationException: Referential integrity constraint violation: "fk_pays: public.ville FOREIGN KEY(pays) REFERENCES public.pays(nom)"; SQL statement:

alter table ville add constraint fk_pays foreign key(pays) references pays on delete cascad [23506-214]
' occurred


###  Effacez les Pays 

In [89]:
query="""

"""
execute(connection,query,show=True)

0 rows


<cursor object at 0x000001D112B94740; closed: -1>

In [90]:
query="""
select * from Pays
"""
execute(connection,query,show=True)

The error 'Table "pays" not found; SQL statement:

select * from Pays
 [42102-214]
DETAIL:  org.h2.jdbc.JdbcSQLSyntaxErrorException: Table "pays" not found; SQL statement:

select * from Pays
 [42102-214]
' occurred


### Exécutez la requête suivante; pourquoi Paris a disparu ?

In [86]:
query="""
select * from Ville
"""
execute(connection,query,show=True)

1 rows, display only 10 rows
|nom  |population|pays  |
|------------------------
|Paris|3000000   |France|


<cursor object at 0x000001D112B94200; closed: -1>

In [91]:
show_schema(connection)

*********** Tables ************
1 rows, display only 1000 rows
|table_name|
|-----------
|ville     |


*********** Domaines ************
0 rows


*********** Attributs ************
3 rows, display only 1000 rows
|table_name|column_name|data_type        |column_default|is_nullable|character_maximum_length|numeric_precision|
|---------------------------------------------------------------------------------------------------------------
|ville     |nom        |character varying|None          |NO         |50                      |None             |
|ville     |population |numeric          |None          |YES        |None                    |10               |
|ville     |pays       |character varying|None          |YES        |50                      |None             |


*********** Contraintes dIntégrité ************
1 rows, display only 1000 rows
|table_name|constraint_name|constraint_type|check_clause|
|--------------------------------------------------------
|ville     |CONSTRAINT_6 

## MISE À JOUR DE TABLES

Considérer le schéma Entreprise du TD précédent rappelé ici :

- EMPLOYE (<u>NumSS</u>, NomE, PrenomE, NumChef*, VilleE, DateNaiss) 
- PROJET(<u>NumProj</u>, NomProj, RespProj*, VilleP, Budget) 
- EMBAUCHE (<u>NumSS*, NumProj*</u>, DateEmb, Profil*) 
- GRILLE_SAL (<u>Profil</u>, salaire)

La clé primaire de chaque relation est soulignée et les attributs des clés étrangères sont suivis d’un astérisque. Cette base contient des informations sur des employés et sur les projets dans lesquels ils sont impliqués. Ces employés sont embauchés dans un projet sur un profil donné et perçoivent un salaire en fonction de ce profil. Le chef d'un employé dans la table Employé et le chef d'un projet dans la table Projet sont des employés.

Le but de cet exercice est d'exprimer des instructions permettant d'insérer, de modifier et de supprimer des nuplets.

Dire à chaque fois si l'instruction exprimée est acceptée ou rejetée par le système en justifiant.

In [92]:
schemafile=open("TME9-creations-H2.sql",mode="r",encoding='utf-8')
create_schema=schemafile.read()
execute(connection,"drop all objects")
execute(connection, create_schema)
connection.commit()

0 rows
0 rows


In [93]:
show_schema(connection)

*********** Tables ************
4 rows, display only 1000 rows
|table_name|
|-----------
|embauche  |
|employe   |
|projet    |
|grille_sal|


*********** Domaines ************
5 rows, display only 1000 rows
|domain_name|check_clause                                 |
|----------------------------------------------------------
|dsal       |VALUE <= 90000                               |
|ddatenaiss |DATEDIFF(YEAR, VALUE, CURRENT_DATE) <= 70    |
|dvilles    |LOWER(VALUE) IN('paris', 'lyon', 'marseille')|
|dnumproj   |CHAR_LENGTH(VALUE) >= 5                      |
|dnumss     |CHAR_LENGTH(VALUE) = 5                       |


*********** Attributs ************
17 rows, display only 1000 rows
|table_name|column_name|data_type        |column_default|is_nullable|character_maximum_length|numeric_precision|
|---------------------------------------------------------------------------------------------------------------
|embauche  |numss      |numeric          |None          |NO         |None    

### INSERTIONS

##### Insérer l'employé identifié par '12456' qui se prénomme 'Alain'.


In [96]:
query="""
insert into employe values(12456,null,'Alain',null,null,null)
"""
execute(connection,query,show=True)



1 rows, display only 10 rows
The error ''NoneType' object is not iterable' occurred


##### Insérer l'employée identifiée par '21456' qui s'appelle 'LARS Anna', qui habite 'Paris' et qui est née le 25-08-1975.

##### Insérer le projet numéro '78143' dénommé 'ORCA' qui s'opère sous la responsabilité de 'Lars Anna' à Paris et qui a pour budget 250 000 euros.

##### Renseigner dans la base les salaires correspondants aux profils suivants : ('Responsable',80000), ('Développeur', 45000), ('Technicien',35000)

##### Renseigner dans la base le fait que 'Alain' a été embauché dans le projet 'ORCA' en tant que testeur en date du 01-04-2014.

##### Renseigner dans la base le fait que 'LARS Anna' fut embauchée dans le projet 'MEDUSA' en date du 28-02-2012 en tant que 'Développeur'.

### SUPPRESSIONS

##### Supprimer les employés de plus de 67 ans

##### Supprimer les employés prénommés 'Alain'

##### Supprimer l'employée 'LARS Anna'

##### Supprimer les employés embauchés lors de la dernière année.

##### Supprimer les projets dont la proportion des salaires dépasse la moitié du budget.

### MISES À JOUR

##### Désormais on connait que l'employé Alain porte le nom BERNARD. Répercuter cette information.

##### L'employée LARS Anna est promue à la tête du projet identifié par 99245. Elle doit d'abord déménager avant de prendre ses responsabilités. Donner l'instruction qui modifie la ville de cette employée. On suppose qu'il existe un nuplet dans la table Projet avec les données suivantes : (99245,'MEDUSA',null,'Lyon',350000)

##### Le budget de chaque projet équivaut au double des salaires de ses employés. Donner une instruction qui modifie les budgets des projets en conséquence.

# Exercice TME: partie 1


In [ ]:
schemafile=open("TME9-creations-H2.sql",mode="r",encoding='utf-8')
datafile=open("TME9-insertions-H2.sql",mode="r",encoding='utf-8')
create_schema=schemafile.read()
insert_data=datafile.read()
execute(connection,"drop all objects")
execute(connection, create_schema)
execute(connection, insert_data)
connection.commit()

In [ ]:
show_schema(connection)

## Insertions rejetées (voir TME8)

### Proposer une insertion dans la table Employé qui ne respecte pas la contrainte de clé primaire.

In [ ]:
query="""

"""
execute(connection,query,show=True)

### Proposer une insertion dans la table Employe qui ne respecte pas la contrainte de limite d'âge.

In [ ]:
query="""

"""
execute(connection,query,show=True)

### Proposer une insertion dans la table Employe qui ne respecte pas la contrainte de longueur de l'attribut NumSS.

In [ ]:
query="""

"""
execute(connection,query,show=True)

### Proposer une insertion dans la table Employe qui ne respecte pas la contrainte sur les villes possibles.

In [ ]:
query="""

"""
execute(connection,query,show=True)

### Insérer dans la table Employé deux employés avec le même nom et le même prénom.

In [ ]:
query="""

"""
execute(connection,query,show=True)

### Proposer une insertion dans la table Grille_SAL qui ne respecte pas la contrainte sur le salaire qui ne doit pas dépasser 90 000.

In [ ]:
query="""

"""
execute(connection,query,show=True)

### Proposer une insertion dans la table Projet qui ne respecte pas la contrainte référentielle vers Employe : insérer un responsable de projet qui n'est pas dans la table Employé

In [ ]:
query="""

"""
execute(connection,query,show=True)

### Proposer une insertion dans la table Embauche qui ne respecte pas une des contraintes référentielles : par exemple, associer un employé existant à un projet qui n'existe pas.

In [ ]:
query="""

"""
execute(connection,query,show=True)

## Suppressions rejetées

In [ ]:
schemafile=open("TME9-creations-H2.sql",mode="r",encoding='utf-8')
datafile=open("TME9-insertions-H2.sql",mode="r",encoding='utf-8')
create_schema=schemafile.read()
insert_data=datafile.read()
execute(connection,"drop all objects")
execute(connection, create_schema)
execute(connection, insert_data)
connection.commit()

In [ ]:
query="""
select * from Employe e
"""
execute(connection,query,show=True)

### Proposer une suppression de nuplets de Employe qui respecte les contraintes d'intégrité.

In [ ]:
query="""
delete from employe where numss=11122
"""
execute(connection,query,show=True)

### Proposer une suppression de nuplets de Employe qui ne respecte pas une contrainte référentielle d'une autre table. Précisez laquelle puis essayer de contourner les contraintes jusqu'à pouvoir supprimer l'Employe référencé.

In [ ]:
query="""
delete from employe where numss=22334
"""
execute(connection,query,show=True)

### Même question pour Projet.

In [ ]:
query="""

"""
execute(connection,query,show=True)

### Même question pour Grille_sal.

In [ ]:
query="""

"""
execute(connection,query,show=True)

### Est-ce possible de supprimer les nuplets de Embauche sans rencontrer les mêmes problèmes que précédemment ?

In [ ]:
query="""

"""
execute(connection,query,show=True)

## Mises à jour rejetées

### Proposez un mise à jour de Employe qui ne respecte pas les contraintes référentielles puis tenter de trouver une mise-a-jour qui les respecte toutes.

In [ ]:
query="""

"""
execute(connection,query,show=True)

### Même question avec Projet.

In [ ]:
query="""

"""
execute(connection,query,show=True)

### Proposez une mise-a-jour de Projet qui ne respecte pas la contrainte resprojet selon laquelle le responsable d'un projet habite la ville du projet dont il est responsable.

In [ ]:
query="""

"""
execute(connection,query,show=True)

### Proposez une mise-a-jour de profil dans Grille_sal qui ne respecte pas les contraintes référentielles.

In [ ]:
query="""

"""
execute(connection,query,show=True)

## Exercie TME: Partie 2

Le but de cette partie d'illustrer un cas où les mises-a-jour ne sont pas rejetées mais plutôt propagées.


In [ ]:
schemafile=open("TME9-creations-cascade-H2.sql",mode="r",encoding='utf-8')
datafile=open("TME9-insertions-H2.sql",mode="r",encoding='utf-8')
create_schema=schemafile.read()
insert_data=datafile.read()
execute(connection,"drop all objects")
execute(connection, create_schema)
execute(connection, insert_data)
connection.commit()

In [ ]:
query="""
select * from Employe
"""
execute(connection,query,show=True)

## Mises-à-jour

### Reéxécutez les mise-à-jour et les suppressions de la Partie 1 et verifiez ce qui se passe.

### Mettez à jour le numéro d'un employé reférencé par d'autres nuplets

In [ ]:
query="""
update Employe set ....
"""
execute(connection,query,show=True)

In [ ]:
query="""
select * from Employe
"""
execute(connection,query,show=True)

## Suppression

### Supprimer les nuplets de Employe et vérifier que tous les nuplets des autres tables qui référencent des employes venant d'être supprimés seront supprimés eux aussi.

In [ ]:
query="""
delete from Employe ....
"""
execute(connection,query,show=True)

In [ ]:
query="""
select * from Employe
"""
execute(connection,query,show=True)

# Fermer la connexion

In [ ]:
connection.commit() # implicit avec close
connection.close()

In [ ]:
connection